# Evaluation & Ablation Studies

Load trained models, run inference on the validation set, compute CER/WER metrics,
generate a confusion matrix, and run ablation comparisons across model variants.

## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

from src.config import get_default_config
from src.data.loader import load_willett_dataset
from src.data.dataset import create_dataloaders
from src.models.gru_decoder import GRUDecoder
from src.models.cnn_lstm import CNNLSTM
from src.models.transformer import TransformerDecoder
from src.models.cnn_transformer import CNNTransformer
from src.decoding.greedy import greedy_decode
from src.decoding.beam_search import beam_search_decode
from src.evaluation.metrics import compute_cer, compute_wer, exact_match_accuracy

## 2. Load Models

Instantiate each decoder architecture. If saved weights are available, load them;
otherwise use randomly initialized weights (useful for verifying the evaluation pipeline).

In [ ]:
cfg = get_default_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_configs = {
    'GRU': GRUDecoder,
    'CNN-LSTM': CNNLSTM,
    'Transformer': TransformerDecoder,
    'CNN-Transformer': CNNTransformer,
}

models = {}
for name, ModelClass in model_configs.items():
    model = ModelClass(
        input_size=cfg.n_channels,
        hidden_size=cfg.hidden_size,
        num_layers=cfg.num_layers,
        num_classes=cfg.num_classes,
    ).to(device)

    # Try loading saved weights
    weight_path = os.path.join('..', 'checkpoints', f'{name.lower().replace("-", "_")}_best.pt')
    if os.path.exists(weight_path):
        model.load_state_dict(torch.load(weight_path, map_location=device))
        print(f"{name}: loaded weights from {weight_path}")
    else:
        print(f"{name}: using randomly initialized weights")

    model.eval()
    models[name] = model

## 3. Run Inference

Run each model on the validation set and collect predictions.

In [ ]:
trial_index = load_willett_dataset(cfg)
_, val_loader = create_dataloaders(trial_index, cfg)

all_predictions = {name: [] for name in models}
all_targets = []
targets_collected = False

for name, model in models.items():
    for batch in val_loader:
        signals = batch['signals'].to(device)
        with torch.no_grad():
            log_probs = model(signals)

        for i in range(signals.shape[0]):
            decoded = greedy_decode(log_probs[i])
            all_predictions[name].append(decoded)
            if not targets_collected:
                all_targets.append(batch['targets'][i])

    targets_collected = True
    print(f"{name}: decoded {len(all_predictions[name])} samples")

## 4. Compute Metrics (CER/WER)

Calculate Character Error Rate and Word Error Rate for each model.

In [ ]:
metrics_table = []

for name in models:
    preds = all_predictions[name]
    cer = compute_cer(preds, all_targets)
    wer = compute_wer(preds, all_targets)
    em = exact_match_accuracy(preds, all_targets)

    metrics_table.append({
        'Model': name,
        'CER (%)': round(cer * 100, 2),
        'WER (%)': round(wer * 100, 2),
        'Exact Match (%)': round(em * 100, 2),
    })
    print(f"{name:20s}  CER={cer*100:.2f}%  WER={wer*100:.2f}%  EM={em*100:.2f}%")

metrics_df = pd.DataFrame(metrics_table)
metrics_df

## 5. Confusion Matrix

Build a character-level confusion matrix from the best model's predictions.

In [ ]:
from collections import Counter

# Use the first model's predictions for the confusion matrix
best_model_name = metrics_df.loc[metrics_df['CER (%)'].idxmin(), 'Model']
preds = all_predictions[best_model_name]

vocab = list('abcdefghijklmnopqrstuvwxyz ')
vocab_idx = {c: i for i, c in enumerate(vocab)}
n_vocab = len(vocab)

conf_matrix = np.zeros((n_vocab, n_vocab), dtype=int)
for pred, target in zip(preds, all_targets):
    target_str = target if isinstance(target, str) else str(target)
    min_len = min(len(pred), len(target_str))
    for j in range(min_len):
        t_char = target_str[j].lower()
        p_char = pred[j].lower()
        if t_char in vocab_idx and p_char in vocab_idx:
            conf_matrix[vocab_idx[t_char], vocab_idx[p_char]] += 1

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(conf_matrix, cmap='Blues', aspect='auto')
ax.set_xticks(range(n_vocab))
ax.set_yticks(range(n_vocab))
ax.set_xticklabels(vocab, fontsize=8)
ax.set_yticklabels(vocab, fontsize=8)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Character Confusion Matrix ({best_model_name})')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 6. Ablation Comparison

Compare CER across models with different configurations (e.g., greedy vs beam search).

In [ ]:
# Compare greedy vs beam search decoding on the best model
best_model = models[best_model_name]

greedy_preds = []
beam_preds = []

for batch in val_loader:
    signals = batch['signals'].to(device)
    with torch.no_grad():
        log_probs = best_model(signals)

    for i in range(signals.shape[0]):
        greedy_preds.append(greedy_decode(log_probs[i]))
        beam_preds.append(beam_search_decode(log_probs[i], beam_width=5))

greedy_cer = compute_cer(greedy_preds, all_targets)
beam_cer = compute_cer(beam_preds, all_targets)

print(f"Model: {best_model_name}")
print(f"  Greedy decode CER: {greedy_cer*100:.2f}%")
print(f"  Beam search  CER: {beam_cer*100:.2f}%")
print(f"  Improvement:       {(greedy_cer - beam_cer)*100:.2f} pp")

## 7. Results Summary Table

In [ ]:
# Display final comparison
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(metrics_df))
ax.bar(x, metrics_df['CER (%)'], color='steelblue', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Model'], rotation=15)
ax.set_ylabel('CER (%)')
ax.set_title('Character Error Rate by Model')
for i, v in enumerate(metrics_df['CER (%)']):
    ax.text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print("\nFull results:")
print(metrics_df.to_string(index=False))

---

**Next:** Proceed to `05b_neural_representations.ipynb` for latent space analysis.